# CV2 Final Project: Illumination-Aware Product Photography

**Repository:** `https://github.com/hj2713/CV2.git`  
**Project path inside repository:** `Project/Code/`

## Purpose
This notebook is the primary Google Colab workflow for the project. It runs a complete pipeline for illumination-aware product photography:

1. Segment the product with Segment Anything (SAM).
2. Estimate the foreground lighting direction with Sobel-shading and a DPT-Large baseline.
3. Create three clean studio composite variants with deterministic shadow synthesis.
4. Evaluate outputs with SDCS and LPIPS.
5. Save versioned results and optionally push them back to GitHub.

## Running Instructions
Run cells sequentially from top to bottom. Cell 1.6 discovers every supported image in `data/raw_images/`; for testing, keep only the desired test images in that folder.

## Runtime Requirement
Use a GPU runtime in Colab: `Runtime > Change runtime type > Hardware accelerator > T4 GPU`.

---
# Section 1: Environment Setup
Run all setup cells at the start of each Colab session. Cell 1.5 is required because it configures Drive-backed model caching for SAM and DPT-Large.

In [1]:
# ============================================================
# Cell 1.1: Clone or Refresh Project Files from GitHub
# ============================================================
# This bootstrap cell cannot import constants.py until the repository has
# been cloned. These values mirror Code/constants.py.

import os
import subprocess

BOOTSTRAP_REPOSITORY_URL = "https://github.com/hj2713/CV2.git"
BOOTSTRAP_COLAB_REPOSITORY_DIR = "/content/cv2_repo"
BOOTSTRAP_SPARSE_CHECKOUT_PATH = "Project"
BOOTSTRAP_BRANCH = "main"


def run_command(command, cwd=None, check=True):
    result = subprocess.run(command, cwd=cwd, capture_output=True, text=True)
    if check and result.returncode != 0:
        raise RuntimeError((result.stderr or result.stdout).strip())
    return result


def refresh_checkout():
    print("Fetching latest Project/ files from GitHub.")
    run_command(["git", "fetch", "origin", BOOTSTRAP_BRANCH], cwd=BOOTSTRAP_COLAB_REPOSITORY_DIR)
    run_command(["git", "sparse-checkout", "init", "--cone"], cwd=BOOTSTRAP_COLAB_REPOSITORY_DIR)
    run_command(["git", "sparse-checkout", "set", BOOTSTRAP_SPARSE_CHECKOUT_PATH], cwd=BOOTSTRAP_COLAB_REPOSITORY_DIR)
    run_command(["git", "reset", "--hard", f"origin/{BOOTSTRAP_BRANCH}"], cwd=BOOTSTRAP_COLAB_REPOSITORY_DIR)
    run_command(["git", "checkout", "-B", BOOTSTRAP_BRANCH, f"origin/{BOOTSTRAP_BRANCH}"], cwd=BOOTSTRAP_COLAB_REPOSITORY_DIR)
    commit = run_command(["git", "rev-parse", "--short", "HEAD"], cwd=BOOTSTRAP_COLAB_REPOSITORY_DIR).stdout.strip()
    print(f"Repository refreshed to origin/{BOOTSTRAP_BRANCH} at commit {commit}.")


if not os.path.exists(BOOTSTRAP_COLAB_REPOSITORY_DIR):
    print(f"Cloning repository from {BOOTSTRAP_REPOSITORY_URL}.")
    clone_result = run_command([
        "git", "clone", "--filter=blob:none", "--no-checkout",
        BOOTSTRAP_REPOSITORY_URL, BOOTSTRAP_COLAB_REPOSITORY_DIR,
    ])
    refresh_checkout()
    print("Repository cloned with sparse checkout for Project/.")
else:
    print(f"Repository already exists at {BOOTSTRAP_COLAB_REPOSITORY_DIR}.")
    refresh_checkout()

project_dir = os.path.join(BOOTSTRAP_COLAB_REPOSITORY_DIR, BOOTSTRAP_SPARSE_CHECKOUT_PATH)
print(f"Project directory: {project_dir}")
print("Top-level project contents:")
for item in sorted(os.listdir(project_dir)):
    print(f"  {item}")

Cloning repository from https://github.com/hj2713/CV2.git.
Fetching latest Project/ files from GitHub.
Repository refreshed to origin/main at commit 2674954.
Repository cloned with sparse checkout for Project/.
Project directory: /content/cv2_repo/Project
Top-level project contents:
  .gitignore
  Code
  Details about project and code


In [ ]:
# ============================================================
# Cell 1.2: Set Working Directory and Import Notebook Helpers
# ============================================================

# This cell moves Colab into the project Code/ folder and makes the
# project modules importable. Run it after Cell 1.1 refreshes the repo.

import os
import sys

CODE_DIR = "/content/cv2_repo/Project/Code"

if not os.path.exists(CODE_DIR):
    raise FileNotFoundError("Code directory not found. Run Cell 1.1 first.")

os.chdir(CODE_DIR)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

from notebook_helpers import print_colab_code_directory

print_colab_code_directory()


In [ ]:
# ============================================================
# Cell 1.3: Install Python Dependencies
# ============================================================

# Installs Python packages listed in requirements.txt and prints GPU status.
# Keep this as a separate cell so dependency errors are easy to diagnose.

from notebook_helpers import install_requirements

install_requirements()


In [ ]:
# ============================================================
# Cell 1.4: Create Data Directory Structure
# ============================================================

# Creates input, mask, output, and local fallback cache directories.
# It is safe to rerun; existing files are not deleted.

from notebook_helpers import create_data_directories

create_data_directories()


In [ ]:
# ============================================================
# Cell 1.5: Mount Google Drive and Configure Model Cache
# ============================================================

# Mounts Google Drive and points HuggingFace/SAM cache paths to Drive.
# This prevents large model downloads from repeating after every Colab restart.

from notebook_helpers import configure_drive_cache

CACHE_ENVIRONMENT = configure_drive_cache()


## Image Discovery

Run Cell 1.6 before the module validation cells and before the batch pipeline. The cell scans every supported image already present in `data/raw_images/`. It does not open an upload dialog and it does not assume a fixed number of images.

For a small test run, keep only the desired test images in `data/raw_images/`. For the final run, place the full benchmark set in the same folder and rerun Cell 1.6.


In [ ]:
# ============================================================
# Cell 1.6: Discover Product Images from data/raw_images
# ============================================================

# Finds every supported image currently in data/raw_images/.
# For testing, manually keep fewer files in that folder; for final runs, place
# the full benchmark set there. The first image is copied to test.jpg for
# module-level validation cells, while batch runs exclude test.jpg.

from notebook_helpers import discover_product_images

PIPELINE_IMAGES, PIPELINE_IMAGE_PATHS, TEST_IMAGE_SOURCE = discover_product_images()


---
# Section 2: Module-Level Validation
Run these cells in order before the full benchmark. They verify segmentation, illumination estimation, generation, and evaluation on `test.jpg`.

In [ ]:
# ============================================================
# Cell 2.1: Download or Restore SAM Weights, Then Segment test.jpg
# ============================================================

# Verifies that SAM weights are available, restores/downloads them if needed,
# saves them to Drive, and runs the segmentation module on data/raw_images/test.jpg.

from notebook_helpers import validate_sam_segmentation

validate_sam_segmentation()


In [ ]:
# ============================================================
# Cell 2.1b: Visualize SAM Segmentation Result
# ============================================================

# Displays the original test image beside the SAM product cutout.
# Inspect this before trusting later compositing or metrics.

from notebook_helpers import visualize_sam_segmentation

visualize_sam_segmentation()


In [ ]:
# ============================================================
# Cell 2.2: Validate Sobel-Shading and DPT-Large Estimators
# ============================================================

# Runs the proposed Sobel-shading estimator and the DPT-Large baseline on the
# test image. It also prints Drive cache information for DPT.

from notebook_helpers import validate_illumination_estimators

validate_illumination_estimators()


In [ ]:
# ============================================================
# Cell 2.3: Validate Deterministic Background Compositing
# ============================================================

# Validates deterministic compositing. This uses masks and estimated angles to
# generate a background and shadow; it does not call Stable Diffusion.

from notebook_helpers import run_python_script

print("Running deterministic compositor validation.")
print("This module does not download or load Stable Diffusion weights.")
print("It uses the SAM mask and estimated Sobel angle to synthesize a clean background and cast shadow.")
run_python_script("core/3_generation.py")


In [ ]:
# ============================================================
# Cell 2.4: Validate SDCS Evaluation Metric
# ============================================================
# SDCS is a heuristic shadow-direction metric. Negative or missing values can
# Checks the SDCS evaluator on the current generated test output.
# SDCS is a shadow-direction heuristic, so unusual shadows can produce low values.

# happen when the generated image has no clear shadow or contradicts the prompt.

from notebook_helpers import run_python_script

run_python_script("core/4_evaluation.py")


---
# Section 3: Full Benchmark Pipeline
The batch script processes every supported image in `data/raw_images/`, excluding `test.jpg`. Output directories and CSV files are versioned so previous runs are preserved.

In [ ]:
# ============================================================
# Cell 3.1: Determine Versioned Run ID
# ============================================================

# Creates the next versioned output location, for example run_003 and
# results_003.csv, so previous Colab results are not overwritten.

from notebook_helpers import prepare_run_id

RUN_ID, RUN_OUTDIR, RUN_CSV = prepare_run_id()


In [ ]:
# ============================================================
# Cell 3.2: Run Full Batch Pipeline
# ============================================================

# Executes main_pipeline.py for every supported source image in data/raw_images/.
# The helper modules are reloaded here so Colab picks up recent Git pulls without
# requiring a runtime restart.

import importlib

import notebook_helpers.scripts as helper_scripts
import notebook_helpers.runs as helper_runs

importlib.reload(helper_scripts)
importlib.reload(helper_runs)

if "RUN_ID" not in globals():
    raise RuntimeError("RUN_ID is not set. Run Cell 3.1 first.")

helper_runs.run_batch_pipeline(RUN_ID)


In [ ]:
# ============================================================
# Cell 3.3: Summarize Results for Current Run
# ============================================================

# Prints the current run CSV as a compact table and reports average metrics.
# Use this table for report numbers after the final batch run.

from notebook_helpers import summarize_run_results

if "RUN_ID" not in globals() or "RUN_CSV" not in globals():
    raise RuntimeError("RUN_ID or RUN_CSV is not set. Run Cells 3.1 and 3.2 first.")

summarize_run_results(RUN_ID, RUN_CSV)


---
# Section 4: Visualization and Demo
Use these cells after a successful batch run to inspect qualitative results and launch the interactive Gradio demo.

In [ ]:
# ============================================================
# Cell 4.1: Display Product-by-Product Diagnostic Rows
# ============================================================

# Builds product-by-product diagnostic rows: original image, SAM cutout,
# naive composite, Sobel-conditioned composite, and DPT-conditioned composite.
# This is the main visual inspection cell for presentation and debugging.

from notebook_helpers import display_diagnostic_rows

if "RUN_ID" not in globals() or "RUN_OUTDIR" not in globals():
    raise RuntimeError("RUN_ID or RUN_OUTDIR is not set. Run Cell 3.1 and Cell 3.2 first.")

DIAGNOSTIC_ROW_PATHS = display_diagnostic_rows(RUN_ID, RUN_OUTDIR)


In [ ]:
# ============================================================
# Cell 4.2: Launch Interactive Gradio Demo
# ============================================================
# This cell runs until interrupted. Use the public Gradio URL for presentation.
# Starts the optional interactive Gradio demo. This cell runs continuously until
# interrupted, so run it only after validating the batch pipeline.


from notebook_helpers import run_python_script

print("Launching Gradio demo. Watch for the public URL in the output.")
run_python_script("app.py")


---
# Section 5: Save Results to GitHub
This section stages only the current notebook, generated masks, and generated outputs. Model checkpoints, HuggingFace caches, and unrelated project files are not staged.

In [ ]:
# ============================================================
# Cell 5.1: Commit and Push Notebook Results
# ============================================================

# Stages only notebook/results artifacts and pushes them to GitHub.
# Model checkpoints and cache folders are intentionally excluded.

from getpass import getpass

from notebook_helpers import commit_and_push_results

GITHUB_TOKEN = getpass("GitHub personal access token (repo scope): ").strip()
commit_and_push_results(GITHUB_TOKEN, RUN_ID if "RUN_ID" in globals() else None)
